# Customer Churn Prediction - ML Classification Project
## Telecom Customer Churn Classification: Logistic Regression vs Random Forest

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, roc_curve, confusion_matrix,
                             classification_report, precision_recall_curve)
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
print("Libraries imported successfully!")

In [ ]:
# Load Data
df = pd.read_csv('../data/telco_churn.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Data Overview
print("Data Info:")
print(df.info())
print("\n" + "="*50)
print("Missing values:")
print(df.isnull().sum())
print("\n" + "="*50)
print("Statistical Summary:")
print(df.describe())

In [ ]:
# Class Distribution Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar plot
churn_counts = df['churn'].value_counts()
axes[0].bar(['No Churn (0)', 'Churn (1)'], churn_counts.values, color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Churn Status')
axes[0].set_ylabel('Count')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(churn_counts.values, labels=['No Churn', 'Churn'], autopct='%1.1f%%',
           colors=['#2ecc71', '#e74c3c'], explode=(0.05, 0.05), shadow=True)
axes[1].set_title('Class Distribution Percentage', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../images/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Class 0 (No Churn): {churn_counts[0]} ({churn_counts[0]/len(df)*100:.1f}%)")
print(f"Class 1 (Churn): {churn_counts[1]} ({churn_counts[1]/len(df)*100:.1f}%)")

In [ ]:
# Correlation Matrix
plt.figure(figsize=(12, 8))
correlation = df.corr()
mask = np.triu(np.ones_like(correlation, dtype=bool))
sns.heatmap(correlation, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Top correlations with churn
churn_corr = correlation['churn'].sort_values(ascending=False)
print("\nTop 5 features correlated with churn:")
print(churn_corr[1:6])

In [ ]:
# Feature Engineering
X = df.drop('churn', axis=1)
y = df['churn']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")
print(f"Number of features: {X_train.shape[1]}")

In [ ]:
# Model 1: Logistic Regression
print("="*60)
print("MODEL 1: LOGISTIC REGRESSION")
print("="*60)

lr_model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_cv_scores = cross_val_score(lr_model, X_train_scaled, y_train, cv=cv, scoring='roc_auc')
print(f"\n5-Fold Cross-Validation ROC-AUC scores: {lr_cv_scores}")
print(f"Mean CV ROC-AUC: {lr_cv_scores.mean():.4f} (+/- {lr_cv_scores.std()*2:.4f})")

# Train on full training set
lr_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_lr = lr_model.predict(X_test_scaled)
y_pred_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
# Model 2: Random Forest
print("\n" + "="*60)
print("MODEL 2: RANDOM FOREST")
print("="*60)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)

# Cross-validation
rf_cv_scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=cv, scoring='roc_auc')
print(f"\n5-Fold Cross-Validation ROC-AUC scores: {rf_cv_scores}")
print(f"Mean CV ROC-AUC: {rf_cv_scores.mean():.4f} (+/- {rf_cv_scores.std()*2:.4f})")

# Train on full training set
rf_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_rf = rf_model.predict(X_test_scaled)
y_pred_proba_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
# Evaluation Function
def evaluate_model(y_test, y_pred, y_pred_proba, model_name):
    metrics = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_pred_proba)
    }
    
    print(f"\n{model_name} Performance Metrics:")
    print("-" * 40)
    for metric, value in metrics.items():
        print(f"{metric:12s}: {value:.4f}")
    
    return metrics

# Evaluate both models
lr_metrics = evaluate_model(y_test, y_pred_lr, y_pred_proba_lr, "Logistic Regression")
rf_metrics = evaluate_model(y_test, y_pred_rf, y_pred_proba_rf, "Random Forest")

In [ ]:
# Confusion Matrices Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, (y_pred, title) in enumerate([(y_pred_lr, 'Logistic Regression'), (y_pred_rf, 'Random Forest')]):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False,
                xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
    axes[idx].set_title(f'{title}\nConfusion Matrix', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('../images/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC Curves
plt.figure(figsize=(10, 8))

# Logistic Regression ROC
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_pred_proba_lr)
auc_lr = roc_auc_score(y_test, y_pred_proba_lr)
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {auc_lr:.3f})', linewidth=2)

# Random Forest ROC
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_pred_proba_rf)
auc_rf = roc_auc_score(y_test, y_pred_proba_rf)
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {auc_rf:.3f})', linewidth=2)

# Diagonal line
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier (AUC = 0.5)', alpha=0.5)

plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../images/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance (Random Forest)
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(feature_importance)))
plt.barh(feature_importance['feature'], feature_importance['importance'], color=colors)
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.title('Random Forest - Feature Importance', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../images/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 5 Most Important Features:")
print(feature_importance.head())

In [ ]:
# Precision-Recall Curves
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, (y_proba, title, color) in enumerate([
    (y_pred_proba_lr, 'Logistic Regression', 'blue'),
    (y_pred_proba_rf, 'Random Forest', 'green')
]):
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    axes[idx].plot(recall, precision, color=color, linewidth=2)
    axes[idx].set_xlabel('Recall', fontsize=11)
    axes[idx].set_ylabel('Precision', fontsize=11)
    axes[idx].set_title(f'{title}\nPrecision-Recall Curve', fontsize=12, fontweight='bold')
    axes[idx].grid(alpha=0.3)
    axes[idx].fill_between(recall, precision, alpha=0.2, color=color)

plt.tight_layout()
plt.savefig('../images/precision_recall.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Final Comparison Table
comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'Logistic Regression': [lr_metrics[m] for m in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']],
    'Random Forest': [rf_metrics[m] for m in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']]
})

print("\n" + "="*60)
print("FINAL MODEL COMPARISON")
print("="*60)
print(comparison_df.to_string(index=False))

# Determine best model
best_auc = 'Random Forest' if rf_metrics['ROC-AUC'] > lr_metrics['ROC-AUC'] else 'Logistic Regression'
best_f1 = 'Random Forest' if rf_metrics['F1-Score'] > lr_metrics['F1-Score'] else 'Logistic Regression'

print(f"\n🏆 Best Model by ROC-AUC: {best_auc}")
print(f"🏆 Best Model by F1-Score: {best_f1}")
print("\n✅ Recommendation: Random Forest performs better for this churn prediction task")

In [ ]:
# Additional Insights
print("\n" + "="*60)
print("BUSINESS INSIGHTS")
print("="*60)

# Calculate business metrics
tn_lr, fp_lr, fn_lr, tp_lr = confusion_matrix(y_test, y_pred_lr).ravel()
tn_rf, fp_rf, fn_rf, tp_rf = confusion_matrix(y_test, y_pred_rf).ravel()

print(f"\nLogistic Regression:")
print(f"  - Correctly identified churners: {tp_lr}")
print(f"  - Missed churners (False Negatives): {fn_lr}")
print(f"  - False alarms (False Positives): {fp_lr}")

print(f"\nRandom Forest:")
print(f"  - Correctly identified churners: {tp_rf}")
print(f"  - Missed churners (False Negatives): {fn_rf}")
print(f"  - False alarms (False Positives): {fp_rf}")

print(f"\n📊 Key Takeaway: Random Forest reduces missed churners by {fn_lr - fn_rf} compared to Logistic Regression")